# A1redo — Benchmark VLM tier-2 trên Colab T4 (Phước)

Theo `notes/2026-10-29-a1-redo-plan.md` §2B/§2C. **Bắt buộc: Runtime → Change runtime type → T4 GPU.**

- Số đo nào cũng là **T4 numbers**; không nói gì về tốc độ 3050.
- Mỗi model chạy trong **MỘT process python riêng** (bitsandbytes leak giữa các model trong cùng kernel).
- Kết quả ghi thẳng lên Drive → session rớt không mất số đo.
- Thời lượng dự kiến: ~1.5–2 h (gồm tải weight lần đầu).

Nếu bị từ chối T4 (usage limit): thử lại tối nay / sáng mai, hoặc để Huy chạy lại notebook này trên tài khoản của Huy — **không** chuyển sang ước lượng.

In [ ]:
# Cell 1 — xác nhận đúng T4. Không phải T4 -> STOP, reconnect.
!nvidia-smi --query-gpu=name,memory.total,memory.used,clocks.sm,clocks.max.sm,temperature.gpu --format=csv
import torch
name = torch.cuda.get_device_properties(0).name
assert "T4" in name, f"GPU không phải T4: {name} — các số đo sẽ không phải T4 numbers. STOP."
print("GPU OK:", name, round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1), "GB")

In [ ]:
# Cell 2 — mount Drive: kết quả sống sót qua disconnect.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/vlm_a1

In [ ]:
# Cell 3 — code + dữ liệu: clone nhánh A1 (zip 16 cửa sổ đã commit trong repo).
!test -d vlm-anomalies-detection || git clone -b feat/phuoc-a1-vlm-bench https://github.com/Kyosaki6/vlm-anomalies-detection.git
%cd vlm-anomalies-detection
!unzip -qo notes/bench_windows.zip
!ls bench_windows | head

In [ ]:
# Cell 4 — cài deps; GIỮ torch của Colab (cài lại torch làm hỏng CUDA wheels).
!pip install -q -r requirements-vlm.txt
!pip freeze | grep -iE "^(torch|transformers|bitsandbytes|accelerate|num2words|pillow|opencv|pyyaml)" | tee /content/drive/MyDrive/vlm_a1/pip_versions.txt
import transformers
for c in ["Qwen2VLForConditionalGeneration", "Qwen2_5_VLForConditionalGeneration", "SmolVLMForConditionalGeneration"]:
    assert hasattr(transformers, c), f"transformers {transformers.__version__} thiếu class {c}; pip install -U transformers"
print("transformers OK:", transformers.__version__)

HF auth là **tùy chọn** (cả 3 model đều public). Nếu bị rate-limit:
```python
from huggingface_hub import login
login()  # dán read token; KHÔNG bao giờ commit token
```

In [ ]:
# Cell 6 — tải weight vào disk local TRƯỚC khi đo: thời gian tải không tính vào load_seconds.
!huggingface-cli download Qwen/Qwen2-VL-2B-Instruct
!huggingface-cli download Qwen/Qwen2.5-VL-3B-Instruct
!huggingface-cli download HuggingFaceTB/SmolVLM2-2.2B-instruct

In [ ]:
# Cell 7 — DRY RUN trước (redo plan §2C.1): validate windows + đếm token, không đốt GPU.
!python scripts/bench_vlm.py --dry-run --windows notes/vlm-bench-windows.json 2>&1 | tee /content/drive/MyDrive/vlm_a1/log_dryrun.txt

In [ ]:
# Cell 8 — warm GPU 30 s (§2C.2) + log clocks trước/sau. Nếu clocks sau < 80% clocks trước -> model bị throttle, chạy lại model đó.
import subprocess, time, torch
def clocks():
    return subprocess.run(["nvidia-smi", "--query-gpu=clocks.sm,clocks.max.sm,temperature.gpu", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout.strip()
print("clocks BEFORE:", clocks())
a = torch.randn(2048, 2048, device="cuda")
t0 = time.time()
while time.time() - t0 < 30:
    a @ a
print("clocks AFTER :", clocks())
with open("/content/drive/MyDrive/vlm_a1/log_gpu_warmup.txt", "w") as f:
    f.write(f"before: {clocks()}\n")

## Chạy benchmark — mỗi model MỘT cell (MỘT process)

Vòng uncapped: `--warmup 2 --runs 8`, quality pass trên đủ 16 cửa sổ, variant images + video (Qwen) / có + không image splitting (SmolVLM2).

In [ ]:
# Model 1/3 — Qwen2-VL-2B (uncapped)
!python -u scripts/bench_vlm.py --model Qwen/Qwen2-VL-2B-Instruct --windows notes/vlm-bench-windows.json --warmup 2 --runs 8 --out /content/drive/MyDrive/vlm_a1/raw_qwen2vl2b.json --jsonl /content/drive/MyDrive/vlm_a1/qwen2vl2b.jsonl 2>&1 | tee /content/drive/MyDrive/vlm_a1/log_qwen2vl2b.txt

In [ ]:
# Model 2/3 — Qwen2.5-VL-3B (uncapped)
!python -u scripts/bench_vlm.py --model Qwen/Qwen2.5-VL-3B-Instruct --windows notes/vlm-bench-windows.json --warmup 2 --runs 8 --out /content/drive/MyDrive/vlm_a1/raw_qwen25vl3b.json --jsonl /content/drive/MyDrive/vlm_a1/qwen25vl3b.jsonl 2>&1 | tee /content/drive/MyDrive/vlm_a1/log_qwen25vl3b.txt

In [ ]:
# Model 3/3 — SmolVLM2-2.2B (uncapped; variant nosplit + split)
!python -u scripts/bench_vlm.py --model HuggingFaceTB/SmolVLM2-2.2B-instruct --windows notes/vlm-bench-windows.json --warmup 2 --runs 8 --out /content/drive/MyDrive/vlm_a1/raw_smolvlm2.json --jsonl /content/drive/MyDrive/vlm_a1/smolvlm2.jsonl 2>&1 | tee /content/drive/MyDrive/vlm_a1/log_smolvlm2.txt

## Vòng mô phỏng card 6 GB (`--vram-cap-gb 5.5`)

Chỉ 3 cửa sổ, không quality. OOM dưới cap = model không phải 6 GB-viable (không loại model — Colab T4 đã là hạ tầng chính; số này nói về phương án dự phòng 3050).

In [ ]:
!python -u scripts/bench_vlm.py --model Qwen/Qwen2-VL-2B-Instruct --windows notes/vlm-bench-windows.json --vram-cap-gb 5.5 --runs 3 --skip-quality --out /content/drive/MyDrive/vlm_a1/raw_qwen2vl2b_capped.json 2>&1 | tee /content/drive/MyDrive/vlm_a1/log_qwen2vl2b_capped.txt
!python -u scripts/bench_vlm.py --model Qwen/Qwen2.5-VL-3B-Instruct --windows notes/vlm-bench-windows.json --vram-cap-gb 5.5 --runs 3 --skip-quality --out /content/drive/MyDrive/vlm_a1/raw_qwen25vl3b_capped.json 2>&1 | tee /content/drive/MyDrive/vlm_a1/log_qwen25vl3b_capped.txt
!python -u scripts/bench_vlm.py --model HuggingFaceTB/SmolVLM2-2.2B-instruct --windows notes/vlm-bench-windows.json --vram-cap-gb 5.5 --runs 3 --skip-quality --out /content/drive/MyDrive/vlm_a1/raw_smolvlm2_capped.json 2>&1 | tee /content/drive/MyDrive/vlm_a1/log_smolvlm2_capped.txt

In [ ]:
# Cell cuối — kiểm tra đủ file trên Drive rồi ngắt runtime (giải phóng T4).
!ls -la /content/drive/MyDrive/vlm_a1/
!nvidia-smi --query-gpu=name,clocks.sm,temperature.gpu,memory.used --format=csv

## Sau khi chạy xong (laptop, trong repo)

1. Tải `/content/drive/MyDrive/vlm_a1/*` về máy, đổi tên vào `notes/`:
   - `raw_*.json` → `vlm-bench-raw-t4.json` (gộp/ghép header nếu cần), `raw_*_capped.json` → `vlm-bench-raw-t4-capped.json`
   - `log_*.txt` → `vlm-bench-log-t4.txt` (ghép các log), `pip_versions.txt` giữ nguyên
2. Viết lại `notes/vlm-selection.md` theo redo plan §5 (bảng A/B/C/D, verdict 1 dòng theo §4.5, Handoff A2).
3. Commit + push lên `feat/phuoc-a1-vlm-bench`, đăng verdict vào nhóm.

Ràng buộc: headline latency = median post-warmup; 6 GB fit ghi 'simulated, +0.7 GB margin'; không có claim nào thiếu số đo.